In [73]:
import pandas as pd
import re
import csv
import re
import sys
from collections import Counter, defaultdict

df_extract = pd.read_csv(
    "../data/interim/1_2_extract_15_16_concat.csv",
    low_memory=False,
)

df_ND1516 = pd.read_csv(
    "../data/raw/test_regards_citoyens/ND15+16_interventions_hemicycle_rich.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="warn",
)

## Comparaison pnum vs id_syceron

In [74]:
# === Comparaison simple par id_syceron ===

# extraite equivalent id_syceron vs url Pnum (pas toujours le P)
# (pourrait changer le nom puisque pas forcéement p)
P_NUMBER_RE = re.compile(r"#P?(\d+)", re.I)


def extract_pnum(url):
    if not url or pd.isna(url):
        return None
    url = str(url).strip()
    m = P_NUMBER_RE.search(url)
    return m.group(1) if m else None


# Colonnes normalisées
df_extract["id_syceron"] = pd.to_numeric(
    df_extract["id_syceron"], errors="raise"
).astype("Int64")
df_ND1516["pnum"] = pd.to_numeric(
    df_ND1516["source"].apply(extract_pnum), errors="raise"
).astype("Int64")


# Comparaison sur les colonnes normalisées
ids_extract = set(df_extract["id_syceron"].dropna().astype(int))
ids_ND = set(df_ND1516["pnum"].dropna().astype(int))

common = ids_extract & ids_ND
only_extract = ids_extract - ids_ND
only_ND = ids_ND - ids_extract


print("=== RÉSUMÉ GLOBAL (par id_syceron uniquement) ===")
print(f"IDs communs              : {len(common):>10,}")
print(f"Uniquement dans extract  : {len(only_extract):>10,}")
print(f"Uniquement dans ND15-16  : {len(only_ND):>10,}")

print(f"\nTotal IDs extract : {len(ids_extract):>10,}")
print(f"Total IDs ND      : {len(ids_ND):>10,}")

=== RÉSUMÉ GLOBAL (par id_syceron uniquement) ===
IDs communs              :  1,065,985
Uniquement dans extract  :     61,477
Uniquement dans ND15-16  :     43,516

Total IDs extract :  1,127,462
Total IDs ND      :  1,109,501


In [75]:
# Vérification sur un échantillon aléatoire des IDs communs
sample_common = pd.Series(list(common)).sample(10)

df_sample_extract = df_extract[df_extract["id_syceron"].isin(sample_common)][
    ["id_syceron", "texte"]
].rename(columns={"id_syceron": "id"})

df_sample_ND = df_ND1516[df_ND1516["pnum"].isin(sample_common)][
    ["pnum", "intervention"]
].rename(columns={"pnum": "id"})

# Fusion sur l'id commun
df_check = df_sample_extract.merge(df_sample_ND, on="id")

display(df_check)

print(
    "LÉO, NORMAL QUE TU AIES DES 'DOUBLONS'",
    "\nTON FICHIER MARCHE C'EST À CAUSE  DU CHANGEMENT TEXTE DANS ND)",
)

,id,texte,intervention
0,1280354,La parole est à M. Vincent Descoeur.,<p>La parole est à M. Vincent Descoeur.</p>
1,1311231,"Cette évolution législative, d’une part, s’ins...","<p>Cette évolution législative, d'une part, s'..."
2,1517032,"En effet, c’est pour cela que je le rappelais ...","<p>En effet, c'est pour cela que je le rappela..."
3,1546690,Ils investissent le débat public ; ils veulent...,<p>Ils investissent le débat public ; ils veul...
4,2411309,Mais si ! Ça se fait aujourd’hui et ça ne se f...,<p>Mais si ! Ça se fait aujourd'hui et ça ne s...
5,2799301,La parole est à M. Thierry Benoit.,<p>La parole est à M. Thierry Benoit.</p>
6,3070188,(Il est procédé au scrutin.),<p>Il est procédé au scrutin.</p>
7,3144559,Il n’est pas possible de rappeler à longueur d...,<p>Il n'est pas possible de rappeler à longueu...
8,3162909,Malgré leurs multiples demandes d’être associé...,<p>Malgré leurs multiples demandes d'être asso...
9,3363624,"Je suis saisie de deux amendements identiques,...",<p>Je suis saisie de deux amendements identiqu...


LÉO, NORMAL QUE TU AIES DES 'DOUBLONS' 
TON FICHIER MARCHE C'EST À CAUSE  DU CHANGEMENT TEXTE DANS ND)


In [76]:
# Repérage des cas uniquement dans l'un ou l'autre
df_only_ND = df_ND1516[df_ND1516["pnum"].isin(only_ND)]
df_only_extract = df_extract[df_extract["id_syceron"].isin(only_extract)]


# Filtrer pour garder que ceux (avec et) sans intervenants

# Cas ND
# Vérifie les cas où ni "parlementaire" ni "personnalite" ne sont renseignés
mask_no_speaker = df_only_ND["parlementaire"].fillna("").astype(str).str.strip().eq(
    ""
) & df_only_ND["personnalite"].fillna("").astype(str).str.strip().eq("")

df_only_ND_no_speaker = df_only_ND[mask_no_speaker]
df_only_ND_with_speaker = df_only_ND[~mask_no_speaker]


print(
    f"df_only_ND : Lignes sans parlementaire ET sans personnalite : {len(df_only_ND_no_speaker):,} / {len(df_only_ND):,}"
)


# Cas extract
# Vérifie les cas sans infos acteur/orateur/nom dans extract
mask_no_speaker_extract = (
    df_only_extract["id_acteur"].isna()
    & df_only_extract["nom_orateur"].isna()
    & df_only_extract["id_orateur"].isna()
)

df_only_extract_no_speaker = df_only_extract[mask_no_speaker_extract]
df_only_extract_with_speaker = df_only_extract[~mask_no_speaker_extract]

print(
    f"df_only_extract_no_speaker : Lignes sans id_acteur ET sans nom_orateur ET sans id_orateur: {len(df_only_extract_no_speaker):,} / {len(df_only_extract):,}"
)

# Exports
df_only_ND_with_speaker.to_csv("only_ND_with_speaker.csv", index=False)
df_only_extract_with_speaker.to_csv("only_extract_with_speaker.csv", index=False)
# ça si jamais mais plutôt du bruit :
# df_only_ND.to_csv("only_ND.csv", index=False)
# df_only_extract.to_csv("only_extract.csv", index=False)

df_only_ND : Lignes sans parlementaire ET sans personnalite : 44,570 / 45,408
df_only_extract_no_speaker : Lignes sans id_acteur ET sans nom_orateur ET sans id_orateur: 60,274 / 61,843


## Recherche texte pnum/id "absents" vs fichier opposé

In [ ]:
import unicodedata
import re
import html


def normalize_text(texte):
    if not isinstance(texte, str):
        return ""
    # Normaliser les caractères Unicode
    texte = unicodedata.normalize("NFC", texte)
    # Décoder les entités HTML
    texte = html.unescape(texte)
    # Remplacer les balises <exposant>o</exposant> par "°" pour aligner les docs vs ND
    texte = re.sub(r"<exposant>o</exposant>", "°", texte)  # meh change pas grand chose
    # Supprimer les balises HTML/XML > espace (éviter collage de mots)
    texte = re.sub(r"<[^>]+>", " ", texte)
    # Supprimer contenu entre parenthèses
    # NOTE : CHOIX FORT SELON CE QUI VEUT ÊTRE ÉTUDIÉ
    # Supprime des didascalies ("Applaudissements", etc.)
    # mais aussi tout autre contenu entre parenthèses
    # ne gère pas les parenthèses imbriquées mais sont extrêmement rares (parfois sur (e))
    # texte = re.sub(r"\([^()]*\)", "", texte)
    # Uniformiser apostrophes (utile pour regex)
    texte = texte.replace("’", "'").replace("\u02bc", "'")
    # Normaliser les espaces (après unescape(), couvre \xa0, \t, \n)
    # et supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # BONUS
    texte = str(texte).lower()

    return texte


# colonnes normalisées
df_ND1516["intervention_norm"] = (
    df_ND1516.get("intervention", "").fillna("").apply(normalize_text)
)
df_extract["texte_norm"] = df_extract.get("texte", "").fillna("").apply(normalize_text)

# recréer les sous-ensembles et limiter aux cas avec orateur
df_only_ND = df_ND1516[df_ND1516["pnum"].isin(only_ND)].copy()
df_only_extract = df_extract[df_extract["id_syceron"].isin(only_extract)].copy()

mask_ND_speaker = df_only_ND["parlementaire"].fillna("").astype(str).str.strip().ne(
    ""
) | df_only_ND["personnalite"].fillna("").astype(str).str.strip().ne("")
df_only_ND_with_speaker = df_only_ND[mask_ND_speaker].copy()

mask_extract_speaker = (
    df_only_extract["id_acteur"].notna()
    | df_only_extract["id_orateur"].notna()
    | df_only_extract["nom_orateur"].fillna("").astype(str).str.strip().ne("")
)
df_only_extract_with_speaker = df_only_extract[mask_extract_speaker].copy()


def search_snippets_fast(
    source_df, source_text_col, big_target, snippet_len=150, limit_rows=None
):
    rows = source_df if limit_rows is None else source_df.head(limit_rows)
    results = []
    optional_fields = (
        "source",
        "uid",
        "id_syceron",
        "pnum",
        "id_acteur",
        "id_orateur",
        "nom_orateur",
        "parlementaire",
        "personnalite",
    )
    for idx, row in rows.iterrows():
        snippet = row.get(source_text_col, "")[:snippet_len]
        found = bool(snippet) and (snippet in big_target)
        out = {"index": idx, "snippet": snippet, "found": found}
        # Ajouter TOUS les champs optionnels présents dans la ligne
        for field in optional_fields:
            if field in row:
                out[field] = row[field]
        results.append(out)
    # renvoyer results dans l'ordre voulu
    columns_order = [
        "source",
        "uid",
        "id_syceron",
        "pnum",
        "id_acteur",
        "id_orateur",
        "nom_orateur",
        "parlementaire",
        "personnalite",
        "index",
        "snippet",
        "found",
    ]
    # Filtrer pour ne garder que les colonnes présentes dans les résultats
    columns_order = [col for col in columns_order if col in results[0].keys()]

    return pd.DataFrame(results)[columns_order]


# recherche rapide (snippet in big_text)
big_ND = " ".join(df_ND1516["intervention_norm"].dropna().tolist())
big_extract = " ".join(df_extract["texte_norm"].dropna().tolist())

fast_res_extract_in_ND = search_snippets_fast(
    df_only_extract_with_speaker, "texte_norm", big_ND, snippet_len=150, limit_rows=None
)
fast_res_ND_in_extract = search_snippets_fast(
    df_only_ND_with_speaker,
    "intervention_norm",
    big_extract,
    snippet_len=150,
    limit_rows=None,
)

fast_res_extract_in_ND.to_csv(
    "fast_only_extract_with_speaker_search_in_ND.csv", index=False
)
fast_res_ND_in_extract.to_csv(
    "fast_only_ND_with_speaker_search_in_extract.csv", index=False
)

print(
    "fast_only_extract_with_speaker -> found:",
    fast_res_extract_in_ND["found"].sum(),
    "/",
    len(fast_res_extract_in_ND),
)
print(
    "fast_only_ND_with_speaker -> found:",
    fast_res_ND_in_extract["found"].sum(),
    "/",
    len(fast_res_ND_in_extract),
)

# Décompte SANS les cas où source contient "congres" (insensible à la casse)
fast_res_ND_in_extract_no_congres = fast_res_ND_in_extract[
    ~fast_res_ND_in_extract["source"].str.contains("congres", case=False, na=False)
]

print("\n--- Décompte SANS les cas 'congres' ---")
print(
    "fast_only_ND_with_speaker -> found (sans congres):",
    fast_res_ND_in_extract_no_congres["found"].sum(),
    "/",
    len(fast_res_ND_in_extract_no_congres),
)

fast_res_ND_in_extract_no_congres.to_csv(
    "fast_only_ND_with_speaker_search_in_extract_no_congres.csv", index=False
)

fast_only_extract_with_speaker -> found: 899 / 1569
fast_only_ND_with_speaker -> found: 382 / 838

--- Décompte SANS les cas 'congres' ---
fast_only_ND_with_speaker -> found (sans congres): 330 / 478
